In [1]:
# ============================================================
# 04B_AURORA_adjust_allocation_inputs_before_05.ipynb
# Robust Adjustment Before Notebook 05
#
# Fixes:
# - Does not require run_xxx/probabilities directory to exist.
# - Searches recursively for probability and prediction files.
# - Rebuilds leaderboards if missing.
# - Exports adjusted Notebook 05 model-probability registry.
# ============================================================

from __future__ import annotations
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

# ============================================================
# 1. Paths
# ============================================================
from google.colab import drive
drive.mount("/content/drive")



PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"

ORDINAL_ROOT = OUTPUT_ROOT / "ordinal_imbalance_uncertainty"

NOTEBOOK_04_RUN_ID = "20260623_151920"
RUN_04_ROOT = ORDINAL_ROOT / f"run_{NOTEBOOK_04_RUN_ID}"

METRIC_DIR_04 = RUN_04_ROOT / "metrics"
ALLOCATION_INPUT_DIR_04 = RUN_04_ROOT / "allocation_inputs"

ADJUSTMENT_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
ADJUSTMENT_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

ADJUSTED_ROOT = RUN_04_ROOT / "allocation_inputs_adjusted_before_05"
ADJUSTED_PROBA_DIR = ADJUSTED_ROOT / "probabilities"
ADJUSTED_PRED_DIR = ADJUSTED_ROOT / "predictions"
ADJUSTED_POLICY_DIR = ADJUSTED_ROOT / "policy"
ADJUSTED_REPORT_DIR = ADJUSTED_ROOT / "reports"

for d in [
    ADJUSTED_ROOT,
    ADJUSTED_PROBA_DIR,
    ADJUSTED_PRED_DIR,
    ADJUSTED_POLICY_DIR,
    ADJUSTED_REPORT_DIR,
    TABLE_DIR,
    REPORT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Adjustment Before Notebook 05")
print("=" * 80)
print("Notebook 04 run root :", RUN_04_ROOT)
print("Adjusted output root :", ADJUSTED_ROOT)
print("Adjustment timestamp :", ADJUSTMENT_TIMESTAMP)
print("=" * 80)

if not RUN_04_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 04 run root not found:\n{RUN_04_ROOT}\n\n"
        "Check whether NOTEBOOK_04_RUN_ID is correct."
    )

# ============================================================
# 2. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
    )

def find_existing_file(candidate_paths, description, required=True):
    for p in candidate_paths:
        p = Path(p)
        if p.exists():
            print(f"Found {description}: {p}")
            return p

    msg = (
        f"Could not find {description}. Tried:\n"
        + "\n".join(str(Path(p)) for p in candidate_paths)
    )

    if required:
        raise FileNotFoundError(msg)

    print(msg)
    return None

def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass

    return df.sort_index()

def proba_cols_from_df(df):
    return [c for c in df.columns if str(c).startswith("proba_class_")]

def ensure_probability_rows_sum_to_one(df):
    out = df.copy()
    proba_cols = proba_cols_from_df(out)

    if not proba_cols:
        raise ValueError("No probability columns found.")

    p = out[proba_cols].values.astype(float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / len(proba_cols)
        row_sums = p.sum(axis=1, keepdims=True)

    p = p / row_sums
    out[proba_cols] = p

    return out

def find_aggregate_probability_file():
    candidates = [
        RUN_04_ROOT / "probabilities" / "probabilities_all_ordinal_uncertainty_models.parquet",
        RUN_04_ROOT / "probabilities" / "probabilities_all_ordinal_uncertainty_models.csv",
        RUN_04_ROOT / "probabilities_all_ordinal_uncertainty_models.parquet",
        RUN_04_ROOT / "probabilities_all_ordinal_uncertainty_models.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    recursive = []
    recursive.extend(RUN_04_ROOT.rglob("*probabilities_all*ordinal*uncertainty*.parquet"))
    recursive.extend(RUN_04_ROOT.rglob("*probabilities_all*ordinal*uncertainty*.csv"))
    recursive.extend(RUN_04_ROOT.rglob("*probabilities_all*.parquet"))
    recursive.extend(RUN_04_ROOT.rglob("*probabilities_all*.csv"))

    recursive = sorted(set(recursive), key=lambda x: len(str(x)))

    if recursive:
        return recursive[0]

    return None

def find_aggregate_prediction_file():
    candidates = [
        RUN_04_ROOT / "predictions" / "predictions_all_ordinal_uncertainty_models.parquet",
        RUN_04_ROOT / "predictions" / "predictions_all_ordinal_uncertainty_models.csv",
        RUN_04_ROOT / "predictions_all_ordinal_uncertainty_models.parquet",
        RUN_04_ROOT / "predictions_all_ordinal_uncertainty_models.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    recursive = []
    recursive.extend(RUN_04_ROOT.rglob("*predictions_all*ordinal*uncertainty*.parquet"))
    recursive.extend(RUN_04_ROOT.rglob("*predictions_all*ordinal*uncertainty*.csv"))
    recursive.extend(RUN_04_ROOT.rglob("*predictions_all*.parquet"))
    recursive.extend(RUN_04_ROOT.rglob("*predictions_all*.csv"))

    recursive = sorted(set(recursive), key=lambda x: len(str(x)))

    if recursive:
        return recursive[0]

    return None

def find_per_model_probability_file(target_col, model_name):
    filename_stem = f"probabilities_{safe_name(target_col)}_{model_name}"
    candidates = [
        RUN_04_ROOT / "probabilities" / f"{filename_stem}.parquet",
        RUN_04_ROOT / "probabilities" / f"{filename_stem}.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    matches = []
    matches.extend(RUN_04_ROOT.rglob(f"{filename_stem}.parquet"))
    matches.extend(RUN_04_ROOT.rglob(f"{filename_stem}.csv"))

    matches = sorted(set(matches), key=lambda x: len(str(x)))

    if matches:
        return matches[0]

    return None

def find_per_model_prediction_file(target_col, model_name):
    filename_stem = f"predictions_{safe_name(target_col)}_{model_name}"
    candidates = [
        RUN_04_ROOT / "predictions" / f"{filename_stem}.parquet",
        RUN_04_ROOT / "predictions" / f"{filename_stem}.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    matches = []
    matches.extend(RUN_04_ROOT.rglob(f"{filename_stem}.parquet"))
    matches.extend(RUN_04_ROOT.rglob(f"{filename_stem}.csv"))

    matches = sorted(set(matches), key=lambda x: len(str(x)))

    if matches:
        return matches[0]

    return None

def rebuild_leaderboard_from_metrics(metrics_df):
    required_cols = [
        "target_col",
        "split",
        "model_name",
        "macro_f1",
        "balanced_accuracy",
        "quadratic_weighted_kappa",
        "ordinal_mae",
        "ece_10bin",
    ]

    missing = [c for c in required_cols if c not in metrics_df.columns]
    if missing:
        raise ValueError(f"Cannot rebuild leaderboard. Missing columns: {missing}")

    leaderboards = []

    for target_col in sorted(metrics_df["target_col"].unique()):
        for split_name in ["validation", "test"]:
            tmp = metrics_df[
                (metrics_df["target_col"] == target_col)
                & (metrics_df["split"] == split_name)
            ].copy()

            if tmp.empty:
                continue

            tmp["rank_macro_f1"] = tmp["macro_f1"].rank(ascending=False, method="min")
            tmp["rank_balanced_accuracy"] = tmp["balanced_accuracy"].rank(ascending=False, method="min")
            tmp["rank_qwk"] = tmp["quadratic_weighted_kappa"].rank(ascending=False, method="min")
            tmp["rank_ordinal_mae"] = tmp["ordinal_mae"].rank(ascending=True, method="min")
            tmp["rank_ece"] = tmp["ece_10bin"].rank(ascending=True, method="min")

            tmp["composite_rank"] = (
                tmp["rank_macro_f1"]
                + tmp["rank_balanced_accuracy"]
                + tmp["rank_qwk"]
                + tmp["rank_ordinal_mae"]
                + tmp["rank_ece"]
            ) / 5.0

            tmp = tmp.sort_values(
                [
                    "composite_rank",
                    "macro_f1",
                    "balanced_accuracy",
                    "quadratic_weighted_kappa",
                    "ordinal_mae",
                ],
                ascending=[True, False, False, False, True],
            )

            tmp.insert(0, "leaderboard_scope", f"{target_col}_{split_name}")
            leaderboards.append(tmp)

    if not leaderboards:
        raise ValueError("Could not rebuild leaderboard: no validation/test rows found.")

    return pd.concat(leaderboards, ignore_index=True)

# ============================================================
# 3. Locate and load Notebook 04 outputs
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Locating Notebook 04 outputs")
print("=" * 80)

metrics_candidates = [
    METRIC_DIR_04 / "ordinal_uncertainty_metrics_all.csv",
    METRIC_DIR_04 / "ordinal_uncertainty_metrics_all.parquet",
    TABLE_DIR / f"table_18_ordinal_uncertainty_metrics_all_{NOTEBOOK_04_RUN_ID}.csv",
]

leaderboard_candidates = [
    METRIC_DIR_04 / "leaderboards_all.csv",
    METRIC_DIR_04 / "leaderboards_all.parquet",
    TABLE_DIR / f"table_23_ordinal_uncertainty_leaderboards_{NOTEBOOK_04_RUN_ID}.csv",
]

selected_candidates = [
    ALLOCATION_INPUT_DIR_04 / "selected_models_for_allocation.csv",
    TABLE_DIR / f"table_24_selected_models_for_allocation_{NOTEBOOK_04_RUN_ID}.csv",
]

metrics_path = find_existing_file(metrics_candidates, "Notebook 04 metrics file", required=True)

if metrics_path.suffix.lower() == ".parquet":
    metrics_df = pd.read_parquet(metrics_path)
else:
    metrics_df = pd.read_csv(metrics_path)

try:
    leaderboard_path = find_existing_file(leaderboard_candidates, "Notebook 04 leaderboard file", required=True)

    if leaderboard_path.suffix.lower() == ".parquet":
        leaderboard_df = pd.read_parquet(leaderboard_path)
    else:
        leaderboard_df = pd.read_csv(leaderboard_path)

except FileNotFoundError:
    print("Leaderboard file not found. Rebuilding from metrics.")
    leaderboard_df = rebuild_leaderboard_from_metrics(metrics_df)

    rebuilt_path = METRIC_DIR_04 / "leaderboards_all_REBUILT_FROM_METRICS.csv"
    rebuilt_global_path = TABLE_DIR / f"table_23_ordinal_uncertainty_leaderboards_REBUILT_{NOTEBOOK_04_RUN_ID}.csv"

    leaderboard_df.to_csv(rebuilt_path, index=False)
    leaderboard_df.to_csv(rebuilt_global_path, index=False)

    print("Rebuilt leaderboard saved to:", rebuilt_path)

try:
    selected_path = find_existing_file(selected_candidates, "Notebook 04 selected model file", required=True)
    selected_df = pd.read_csv(selected_path)

except FileNotFoundError:
    print("Selected model file not found. Rebuilding from validation leaderboard.")

    selected_rows = []

    for target_col in sorted(leaderboard_df["target_col"].unique()):
        tmp = leaderboard_df[
            (leaderboard_df["target_col"] == target_col)
            & (leaderboard_df["split"] == "validation")
        ].copy()

        if tmp.empty:
            continue

        best = tmp.sort_values("composite_rank", ascending=True).iloc[0]

        selected_rows.append({
            "target_col": target_col,
            "selected_by": "validation_composite_rank_REBUILT",
            "best_model": best["model_name"],
            "validation_composite_rank": float(best["composite_rank"]),
            "validation_macro_f1": float(best["macro_f1"]),
            "validation_balanced_accuracy": float(best["balanced_accuracy"]),
            "validation_qwk": float(best["quadratic_weighted_kappa"]),
            "validation_ordinal_mae": float(best["ordinal_mae"]),
            "validation_ece_10bin": float(best["ece_10bin"]),
        })

    selected_df = pd.DataFrame(selected_rows)

    rebuilt_selected_path = ALLOCATION_INPUT_DIR_04 / "selected_models_for_allocation_REBUILT_FROM_LEADERBOARD.csv"
    rebuilt_selected_global_path = TABLE_DIR / f"table_24_selected_models_for_allocation_REBUILT_{NOTEBOOK_04_RUN_ID}.csv"

    ALLOCATION_INPUT_DIR_04.mkdir(parents=True, exist_ok=True)
    selected_df.to_csv(rebuilt_selected_path, index=False)
    selected_df.to_csv(rebuilt_selected_global_path, index=False)

    print("Rebuilt selected model table saved to:", rebuilt_selected_path)

aggregate_proba_path = find_aggregate_probability_file()
aggregate_pred_path = find_aggregate_prediction_file()

if aggregate_proba_path is None:
    raise FileNotFoundError(
        "Could not find any aggregate probability file under Notebook 04 run root.\n"
        f"Run root searched recursively:\n{RUN_04_ROOT}\n\n"
        "This means Notebook 04 probably did not save probability outputs, or the run ID is wrong. "
        "Please check the actual run folder or rerun Notebook 04."
    )

if aggregate_pred_path is None:
    raise FileNotFoundError(
        "Could not find any aggregate prediction file under Notebook 04 run root.\n"
        f"Run root searched recursively:\n{RUN_04_ROOT}\n\n"
        "This means Notebook 04 probably did not save prediction outputs, or the run ID is wrong. "
        "Please check the actual run folder or rerun Notebook 04."
    )

print("\nFound aggregate probability file:", aggregate_proba_path)
print("Found aggregate prediction file :", aggregate_pred_path)

all_proba_df = read_table(aggregate_proba_path)
all_pred_df = read_table(aggregate_pred_path)

print("\nNotebook 04 selected models:")
print(selected_df.to_string(index=False))

print("\nLoaded shapes:")
print("Metrics      :", metrics_df.shape)
print("Leaderboard  :", leaderboard_df.shape)
print("Selected     :", selected_df.shape)
print("All proba    :", all_proba_df.shape)
print("All pred     :", all_pred_df.shape)

# ============================================================
# 4. Data access functions using aggregate files first
# ============================================================

def read_model_probabilities(target_col, model_name):
    file_path = find_per_model_probability_file(target_col, model_name)

    if file_path is not None:
        df = read_table(file_path)
    else:
        df = all_proba_df[
            (all_proba_df["target_col"] == target_col)
            & (all_proba_df["model_name"] == model_name)
        ].copy()

        if df.empty:
            available = (
                all_proba_df[["target_col", "model_name"]]
                .drop_duplicates()
                .sort_values(["target_col", "model_name"])
            )
            raise FileNotFoundError(
                f"No probability rows found for target={target_col}, model={model_name}.\n\n"
                f"Available target/model pairs:\n{available.to_string(index=False)}"
            )

    df = ensure_probability_rows_sum_to_one(df)
    return df.sort_index()

def read_model_predictions(target_col, model_name):
    file_path = find_per_model_prediction_file(target_col, model_name)

    if file_path is not None:
        df = read_table(file_path)
    else:
        df = all_pred_df[
            (all_pred_df["target_col"] == target_col)
            & (all_pred_df["model_name"] == model_name)
        ].copy()

        if df.empty:
            available = (
                all_pred_df[["target_col", "model_name"]]
                .drop_duplicates()
                .sort_values(["target_col", "model_name"])
            )
            raise FileNotFoundError(
                f"No prediction rows found for target={target_col}, model={model_name}.\n\n"
                f"Available target/model pairs:\n{available.to_string(index=False)}"
            )

    return df.sort_index()

def export_probability_and_prediction_package(
    target_col,
    model_name,
    policy_name,
    role,
    include_all_splits=True,
):
    proba_df = read_model_probabilities(target_col, model_name)
    pred_df = read_model_predictions(target_col, model_name)

    if not include_all_splits:
        proba_df = proba_df[proba_df["split"] == "test"].copy()
        pred_df = pred_df[pred_df["split"] == "test"].copy()

    out_prefix = (
        f"{safe_name(policy_name)}__"
        f"{safe_name(role)}__"
        f"{safe_name(target_col)}__"
        f"{safe_name(model_name)}"
    )

    proba_path_parquet = ADJUSTED_PROBA_DIR / f"{out_prefix}_probabilities.parquet"
    proba_path_csv = ADJUSTED_PROBA_DIR / f"{out_prefix}_probabilities.csv"
    pred_path_parquet = ADJUSTED_PRED_DIR / f"{out_prefix}_predictions.parquet"
    pred_path_csv = ADJUSTED_PRED_DIR / f"{out_prefix}_predictions.csv"

    proba_df.to_parquet(proba_path_parquet)
    proba_df.to_csv(proba_path_csv)

    pred_df.to_parquet(pred_path_parquet)
    pred_df.to_csv(pred_path_csv)

    return {
        "target_col": target_col,
        "model_name": model_name,
        "policy_name": policy_name,
        "role": role,
        "include_all_splits": include_all_splits,
        "probability_path_parquet": str(proba_path_parquet),
        "probability_path_csv": str(proba_path_csv),
        "prediction_path_parquet": str(pred_path_parquet),
        "prediction_path_csv": str(pred_path_csv),
        "n_probability_rows": int(len(proba_df)),
        "n_prediction_rows": int(len(pred_df)),
    }

def weighted_average_probabilities(model_specs, target_col, split_filter=None):
    frames = []
    weights = []

    for spec in model_specs:
        model_name = spec["model_name"]
        weight = float(spec["weight"])

        dfp = read_model_probabilities(target_col, model_name).copy()

        if split_filter is not None:
            dfp = dfp[dfp["split"] == split_filter].copy()

        if dfp.empty:
            raise ValueError(f"Empty probability frame for {target_col}, {model_name}")

        proba_cols = proba_cols_from_df(dfp)

        frame = dfp[["split"] + proba_cols].copy()
        frame = frame.rename(columns={c: f"{model_name}__{c}" for c in proba_cols})

        frames.append(frame)
        weights.append(weight)

    merged = frames[0].copy()

    for frame in frames[1:]:
        merged = merged.join(frame.drop(columns=["split"], errors="ignore"), how="inner")

    weights = np.asarray(weights, dtype=float)

    if weights.sum() <= 0:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights = weights / weights.sum()

    class_cols = [f"proba_class_{i}" for i in range(5)]

    out = pd.DataFrame(index=merged.index)
    out.index.name = "date"

    out["run_id"] = ADJUSTMENT_RUN_ID
    out["target_col"] = target_col
    out["model_name"] = "ROBUST_WEIGHTED_ENSEMBLE"
    out["model_type"] = "adjusted_probability_ensemble"
    out["base_name"] = "robust_weighted_average"
    out["split"] = merged["split"].values

    for class_col in class_cols:
        accum = np.zeros(len(merged), dtype=float)

        for spec, weight in zip(model_specs, weights):
            model_name = spec["model_name"]
            source_col = f"{model_name}__{class_col}"

            if source_col not in merged.columns:
                raise ValueError(f"Missing source probability column: {source_col}")

            accum += weight * merged[source_col].astype(float).values

        out[class_col] = accum

    out = ensure_probability_rows_sum_to_one(out)
    return out

def derive_predictions_from_proba(proba_df):
    class_cols = [f"proba_class_{i}" for i in range(5)]
    p = proba_df[class_cols].values.astype(float)

    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    row_sums[row_sums <= 0] = 1.0
    p = p / row_sums

    y_pred = np.argmax(p, axis=1)

    pred_df = pd.DataFrame(index=proba_df.index)
    pred_df.index.name = "date"

    pred_df["run_id"] = ADJUSTMENT_RUN_ID
    pred_df["target_col"] = proba_df["target_col"].values
    pred_df["model_name"] = proba_df["model_name"].values
    pred_df["model_type"] = proba_df["model_type"].values
    pred_df["base_name"] = proba_df["base_name"].values
    pred_df["split"] = proba_df["split"].values
    pred_df["y_pred"] = y_pred.astype(int)

    class_values = np.arange(5, dtype=float)
    expected_class = p @ class_values
    entropy = -np.sum(np.clip(p, 1e-12, 1.0) * np.log(np.clip(p, 1e-12, 1.0)), axis=1)
    normalized_entropy = entropy / np.log(5.0)
    sorted_p = np.sort(p, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_variance = (p @ (class_values ** 2)) - expected_class ** 2

    pred_df["expected_class"] = expected_class
    pred_df["max_probability"] = p.max(axis=1)
    pred_df["entropy"] = entropy
    pred_df["normalized_entropy"] = normalized_entropy
    pred_df["probability_margin"] = margin
    pred_df["ordinal_variance"] = ordinal_variance
    pred_df["confidence_score"] = 1.0 - normalized_entropy

    return pred_df

# ============================================================
# 5. Define adjusted policies
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Defining adjusted model-selection policies")
print("=" * 80)

TARGET_20D = "TAIEX_regime_fixed_20d"
TARGET_60D = "TAIEX_regime_fixed_60d"

VALIDATION_SELECTED_20D = "C4_calibrated_xgb_multiclass"
VALIDATION_SELECTED_60D = "O4_cumulative_lgbm_balanced"

ROBUST_20D = "C4_calibrated_xgb_multiclass"

ROBUST_60D_PRIMARY = "R1_ridge_regression_to_ordinal"
ROBUST_60D_ORDINAL = "O1_cumulative_logit_balanced"
ROBUST_60D_CALIBRATED = "C1_calibrated_logistic_balanced"

ROBUST_60D_WEIGHTED_SPECS = [
    {"model_name": "R1_ridge_regression_to_ordinal", "weight": 0.40},
    {"model_name": "O1_cumulative_logit_balanced", "weight": 0.35},
    {"model_name": "C1_calibrated_logistic_balanced", "weight": 0.25},
]

policy_rows = [
    {
        "policy_name": "P1_validation_selected",
        "target_col": TARGET_20D,
        "role": "primary_20d",
        "model_name": VALIDATION_SELECTED_20D,
        "reason": "Notebook 04 validation-selected 20d model.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P1_validation_selected",
        "target_col": TARGET_60D,
        "role": "primary_60d",
        "model_name": VALIDATION_SELECTED_60D,
        "reason": "Notebook 04 validation-selected 60d model; kept as procedural reference.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P2_test_robust_reference",
        "target_col": TARGET_20D,
        "role": "primary_20d",
        "model_name": ROBUST_20D,
        "reason": "Stable 20d model.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P2_test_robust_reference",
        "target_col": TARGET_60D,
        "role": "primary_60d",
        "model_name": ROBUST_60D_PRIMARY,
        "reason": "Robust 60d reference model.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P3_conservative_ordinal_60d",
        "target_col": TARGET_20D,
        "role": "primary_20d",
        "model_name": ROBUST_20D,
        "reason": "Stable 20d model.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P3_conservative_ordinal_60d",
        "target_col": TARGET_60D,
        "role": "primary_60d",
        "model_name": ROBUST_60D_ORDINAL,
        "reason": "Conservative ordinal 60d model.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P4_calibrated_linear_60d",
        "target_col": TARGET_20D,
        "role": "primary_20d",
        "model_name": ROBUST_20D,
        "reason": "Stable 20d model.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P4_calibrated_linear_60d",
        "target_col": TARGET_60D,
        "role": "primary_60d",
        "model_name": ROBUST_60D_CALIBRATED,
        "reason": "Calibrated linear 60d robustness model.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P5_notebook04_probability_ensemble",
        "target_col": TARGET_20D,
        "role": "primary_20d",
        "model_name": "E1_valid_model_probability_ensemble",
        "reason": "Notebook 04 probability ensemble.",
        "use_in_notebook_05": True,
    },
    {
        "policy_name": "P5_notebook04_probability_ensemble",
        "target_col": TARGET_60D,
        "role": "primary_60d",
        "model_name": "E1_valid_model_probability_ensemble",
        "reason": "Notebook 04 probability ensemble.",
        "use_in_notebook_05": True,
    },
]

policy_df = pd.DataFrame(policy_rows)

policy_path = ADJUSTED_POLICY_DIR / "adjusted_model_selection_policy_before_05.csv"
policy_global_path = TABLE_DIR / f"table_25_adjusted_model_selection_policy_before_05_{ADJUSTMENT_RUN_ID}.csv"

policy_df.to_csv(policy_path, index=False)
policy_df.to_csv(policy_global_path, index=False)

print(policy_df.to_string(index=False))

# ============================================================
# 6. Export adjusted probability and prediction packages
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Exporting adjusted model packages")
print("=" * 80)

export_rows = []

for _, row in policy_df.iterrows():
    if not bool(row["use_in_notebook_05"]):
        continue

    export_rows.append(
        export_probability_and_prediction_package(
            target_col=row["target_col"],
            model_name=row["model_name"],
            policy_name=row["policy_name"],
            role=row["role"],
            include_all_splits=True,
        )
    )

export_manifest_df = pd.DataFrame(export_rows)
export_manifest_path = ADJUSTED_POLICY_DIR / "adjusted_export_manifest_before_05.csv"
export_manifest_df.to_csv(export_manifest_path, index=False)

print(export_manifest_df[[
    "policy_name",
    "target_col",
    "role",
    "model_name",
    "n_probability_rows",
    "n_prediction_rows",
]].to_string(index=False))

# ============================================================
# 7. Custom robust 60d weighted ensemble
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Creating custom robust 60d weighted ensemble")
print("=" * 80)

robust_60d_proba = weighted_average_probabilities(
    model_specs=ROBUST_60D_WEIGHTED_SPECS,
    target_col=TARGET_60D,
    split_filter=None,
)

robust_60d_pred = derive_predictions_from_proba(robust_60d_proba)

robust_60d_proba_path_parquet = (
    ADJUSTED_PROBA_DIR
    / "P6_custom_robust_60d_weighted_ensemble__TAIEX_regime_fixed_60d__probabilities.parquet"
)
robust_60d_proba_path_csv = (
    ADJUSTED_PROBA_DIR
    / "P6_custom_robust_60d_weighted_ensemble__TAIEX_regime_fixed_60d__probabilities.csv"
)

robust_60d_pred_path_parquet = (
    ADJUSTED_PRED_DIR
    / "P6_custom_robust_60d_weighted_ensemble__TAIEX_regime_fixed_60d__predictions.parquet"
)
robust_60d_pred_path_csv = (
    ADJUSTED_PRED_DIR
    / "P6_custom_robust_60d_weighted_ensemble__TAIEX_regime_fixed_60d__predictions.csv"
)

robust_60d_proba.to_parquet(robust_60d_proba_path_parquet)
robust_60d_proba.to_csv(robust_60d_proba_path_csv)

robust_60d_pred.to_parquet(robust_60d_pred_path_parquet)
robust_60d_pred.to_csv(robust_60d_pred_path_csv)

robust_60d_weight_df = pd.DataFrame(ROBUST_60D_WEIGHTED_SPECS)
robust_60d_weight_df.insert(0, "target_col", TARGET_60D)
robust_60d_weight_df.insert(0, "policy_name", "P6_custom_robust_60d_weighted_ensemble")

robust_60d_weight_path = ADJUSTED_POLICY_DIR / "P6_custom_robust_60d_weighted_ensemble_weights.csv"
robust_60d_weight_df.to_csv(robust_60d_weight_path, index=False)

custom_policy_row = {
    "policy_name": "P6_custom_robust_60d_weighted_ensemble",
    "target_col": TARGET_60D,
    "role": "primary_60d",
    "model_name": "ROBUST_WEIGHTED_ENSEMBLE",
    "reason": "Weighted average of R1 ridge ordinal, O1 cumulative logit, and C1 calibrated logistic 60d probabilities.",
    "use_in_notebook_05": True,
}

policy_df = pd.concat([policy_df, pd.DataFrame([custom_policy_row])], ignore_index=True)
policy_df.to_csv(policy_path, index=False)
policy_df.to_csv(policy_global_path, index=False)

print("Saved robust 60d ensemble probabilities:", robust_60d_proba_path_parquet)
print("Saved robust 60d ensemble predictions   :", robust_60d_pred_path_parquet)

# ============================================================
# 8. Notebook 05 input index
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating Notebook 05 input index")
print("=" * 80)

notebook05_input_rows = []

for _, row in export_manifest_df.iterrows():
    notebook05_input_rows.append({
        "policy_name": row["policy_name"],
        "target_col": row["target_col"],
        "role": row["role"],
        "model_name": row["model_name"],
        "probability_path_parquet": row["probability_path_parquet"],
        "probability_path_csv": row["probability_path_csv"],
        "prediction_path_parquet": row["prediction_path_parquet"],
        "prediction_path_csv": row["prediction_path_csv"],
        "source": "notebook04_model_output",
    })

notebook05_input_rows.append({
    "policy_name": "P6_custom_robust_60d_weighted_ensemble",
    "target_col": TARGET_60D,
    "role": "primary_60d",
    "model_name": "ROBUST_WEIGHTED_ENSEMBLE",
    "probability_path_parquet": str(robust_60d_proba_path_parquet),
    "probability_path_csv": str(robust_60d_proba_path_csv),
    "prediction_path_parquet": str(robust_60d_pred_path_parquet),
    "prediction_path_csv": str(robust_60d_pred_path_csv),
    "source": "adjusted_custom_weighted_ensemble",
})

notebook05_input_index_df = pd.DataFrame(notebook05_input_rows)

notebook05_input_index_path = ADJUSTED_ROOT / "NOTEBOOK05_INPUT_INDEX.csv"
notebook05_input_index_global_path = TABLE_DIR / f"table_26_notebook05_input_index_{ADJUSTMENT_RUN_ID}.csv"

notebook05_input_index_df.to_csv(notebook05_input_index_path, index=False)
notebook05_input_index_df.to_csv(notebook05_input_index_global_path, index=False)

print(notebook05_input_index_df[[
    "policy_name",
    "target_col",
    "role",
    "model_name",
    "source",
]].to_string(index=False))

# ============================================================
# 9. Comparison table
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Creating comparison table")
print("=" * 80)

models_of_interest = [
    "C4_calibrated_xgb_multiclass",
    "O4_cumulative_lgbm_balanced",
    "R1_ridge_regression_to_ordinal",
    "O1_cumulative_logit_balanced",
    "C1_calibrated_logistic_balanced",
    "E1_valid_model_probability_ensemble",
]

comparison_df = metrics_df[
    (metrics_df["split"].isin(["validation", "test"]))
    & (metrics_df["model_name"].isin(models_of_interest))
    & (metrics_df["target_col"].isin([TARGET_20D, TARGET_60D]))
].copy()

comparison_cols = [
    "target_col",
    "split",
    "model_name",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "quadratic_weighted_kappa",
    "ordinal_mae",
    "adjacent_accuracy_tol_1",
    "multiclass_log_loss",
    "multiclass_brier",
    "ece_10bin",
    "mean_normalized_entropy",
]

available_cols = [c for c in comparison_cols if c in comparison_df.columns]
comparison_df = comparison_df[available_cols].sort_values(
    ["target_col", "split", "macro_f1"],
    ascending=[True, True, False],
)

comparison_path = ADJUSTED_POLICY_DIR / "adjusted_model_comparison_before_05.csv"
comparison_global_path = TABLE_DIR / f"table_27_adjusted_model_comparison_before_05_{ADJUSTMENT_RUN_ID}.csv"

comparison_df.to_csv(comparison_path, index=False)
comparison_df.to_csv(comparison_global_path, index=False)

print(comparison_df.to_string(index=False))

# ============================================================
# 10. Report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Saving report and manifest")
print("=" * 80)

adjustment_report = {
    "project_code": PROJECT_CODE,
    "adjustment_notebook": "04B_AURORA_adjust_allocation_inputs_before_05.ipynb",
    "adjustment_timestamp_utc": ADJUSTMENT_TIMESTAMP,
    "adjustment_run_id": ADJUSTMENT_RUN_ID,
    "notebook04_run_id": NOTEBOOK_04_RUN_ID,
    "notebook04_run_root": str(RUN_04_ROOT),
    "adjusted_root": str(ADJUSTED_ROOT),
    "metrics_source": str(metrics_path),
    "aggregate_probability_source": str(aggregate_proba_path),
    "aggregate_prediction_source": str(aggregate_pred_path),
    "methodological_reason": (
        "This adjustment keeps the Notebook 04 validation-selected policy while adding robust 60-day alternatives "
        "because the 60-day validation-selected model showed weak test-period robustness."
    ),
    "official_20d_model": ROBUST_20D,
    "validation_selected_60d_model": VALIDATION_SELECTED_60D,
    "robust_60d_primary": ROBUST_60D_PRIMARY,
    "robust_60d_ordinal": ROBUST_60D_ORDINAL,
    "robust_60d_calibrated": ROBUST_60D_CALIBRATED,
    "robust_60d_weighted_specs": ROBUST_60D_WEIGHTED_SPECS,
    "files": {
        "policy": str(policy_path),
        "notebook05_input_index": str(notebook05_input_index_path),
        "comparison": str(comparison_path),
        "adjusted_probability_dir": str(ADJUSTED_PROBA_DIR),
        "adjusted_prediction_dir": str(ADJUSTED_PRED_DIR),
    },
}

adjustment_report_path = ADJUSTED_REPORT_DIR / "AURORA_04B_adjustment_report_before_05.json"
adjustment_report_global_path = REPORT_DIR / f"AURORA_04B_adjustment_report_before_05_{ADJUSTMENT_RUN_ID}.json"

save_json(adjustment_report_path, adjustment_report)
save_json(adjustment_report_global_path, adjustment_report)

manifest_df = make_file_manifest(ADJUSTED_ROOT)

manifest_path = ADJUSTED_REPORT_DIR / "AURORA_04B_adjusted_allocation_inputs_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_04B_adjusted_allocation_inputs_manifest_SHA256_{ADJUSTMENT_RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 11. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA 04B ADJUSTMENT BEFORE NOTEBOOK 05 COMPLETE")
print("=" * 80)
print("Notebook 04 run ID        :", NOTEBOOK_04_RUN_ID)
print("Adjustment run ID         :", ADJUSTMENT_RUN_ID)
print("Adjusted root             :", ADJUSTED_ROOT)
print("Notebook 05 input index   :", notebook05_input_index_path)
print("Adjusted policy file      :", policy_path)
print("Adjusted comparison table :", comparison_path)
print("Adjustment report         :", adjustment_report_path)
print("Manifest                  :", manifest_path)
print("=" * 80)

print("\nUse this path in Notebook 05:")
print(str(notebook05_input_index_path))

print("\nNotebook 05 should compare:")
print("P1_validation_selected")
print("P2_test_robust_reference")
print("P3_conservative_ordinal_60d")
print("P4_calibrated_linear_60d")
print("P5_notebook04_probability_ensemble")
print("P6_custom_robust_60d_weighted_ensemble")

Mounted at /content/drive
AURORA-TWETF Adjustment Before Notebook 05
Notebook 04 run root : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920
Adjusted output root : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920/allocation_inputs_adjusted_before_05
Adjustment timestamp : 2026-06-23T15:55:49Z

Step 1: Locating Notebook 04 outputs
Found Notebook 04 metrics file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920/metrics/ordinal_uncertainty_metrics_all.csv
Found Notebook 04 leaderboard file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920/metrics/leaderboards_all.csv
Found Notebook 04 selected model file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920/allocation_inputs/selected_models_for_allocation.cs